In [1]:
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
from catboost import CatBoostClassifier
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

random_state = 42
        


In [ ]:
df = pd.read_csv('cleaned_data.csv' ,index_col=0)
df

,id,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn,cust-loyality,family_member,subscription_count
0,0,Male,old,Yes,Yes,29,Yes,No,DSL,Yes,...,No,One year,Yes,Mailed check,60.10,1653.85,No,Loyal,Married With Dependents,3
1,1,Male,old,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Two year,No,Credit card (automatic),69.50,3778.20,No,Very Loyal,Married With Dependents,4
2,2,Male,old,Yes,No,58,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No,Very Loyal,Married,3
3,3,Female,old,No,No,1,Yes,No,Fiber optic,No,...,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes,New,Single,0
4,4,Female,old,No,No,1,Yes,No,Fiber optic,No,...,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes,New,Single,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,594189,Male,old,No,No,57,Yes,Yes,Fiber optic,No,...,Yes,Two year,No,Bank transfer (automatic),97.55,5460.70,No,Very Loyal,Single,3
594190,594190,Female,old,No,No,72,Yes,Yes,DSL,Yes,...,Yes,Two year,No,Bank transfer (automatic),91.95,6782.15,No,Very Loyal,Single,6
594191,594191,Female,old,Yes,No,72,Yes,Yes,No,No,...,No,Two year,No,Credit card (automatic),24.40,1871.90,No,Very Loyal,Married,0
594192,594192,Female,old,No,No,32,Yes,Yes,Fiber optic,No,...,Yes,Month-to-month,Yes,Electronic check,86.00,2847.20,No,Loyal,Single,1


# Summary

In [3]:
# this is a summary for the data
summary = pd.DataFrame({
    "rows": [df.shape[0]],
    "columns": [df.shape[1]],
    "duplicates": [df.duplicated().sum()],
    "missing_values": [int(df.isna().sum().sum())],
})

target_distribution = pd.DataFrame({
    "count": df["churn"].value_counts(),
    "percentage": (df["churn"].value_counts(normalize=True) * 100).round(2),
})

summary , target_distribution

(     rows  columns  duplicates  missing_values
 0  594194       24           0               0,
         count  percentage
 churn                    
 No     460377       77.48
 Yes    133817       22.52)

In [4]:
# Splitting categorical and numerical columns in different lists for later processing & into x, y

X = df.drop(columns=["churn","id"])
y = df["churn"].map({"No": 0, "Yes": 1})

categorical_columns = X.select_dtypes(include=["object", "string"]).columns.tolist()
numeric_columns = X.select_dtypes(include="number").columns.tolist()

pd.DataFrame({
    "categorical_columns": pd.Series(categorical_columns),
    "numeric_columns": pd.Series(numeric_columns),
})

,categorical_columns,numeric_columns
0,gender,tenure
1,seniorcitizen,monthlycharges
2,partner,totalcharges
3,dependents,subscription_count
4,phoneservice,NaN
5,multiplelines,NaN
6,internetservice,NaN
7,onlinesecurity,NaN
8,onlinebackup,NaN
9,deviceprotection,NaN


In [5]:
#summary statistics  for categorical
df.describe(include='object')

,gender,seniorcitizen,partner,dependents,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,churn,cust-loyality,family_member
count,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194,594194
unique,2,2,2,2,2,3,3,2,2,2,2,2,2,3,2,4,2,4,4
top,Female,old,Yes,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,No,Very Loyal,Single
freq,298738,526395,309554,414362,557893,283384,272386,430201,390810,388104,429298,353893,352759,298918,365579,215372,460377,227536,271154


In [6]:
# summary statistics check for numirical
df.describe(include='number')

,id,tenure,monthlycharges,totalcharges,subscription_count
count,594194.000000,594194.000000,594194.000000,594194.000000,594194.000000
mean,297096.500000,36.577258,65.866223,2494.377057,2.053368
std,171529.177262,25.061922,31.067444,2353.916710,1.968652
min,0.000000,1.000000,18.250000,18.800000,0.000000
25%,148548.250000,12.000000,29.900000,639.650000,0.000000
50%,297096.500000,35.000000,74.100000,1433.650000,2.000000
75%,445644.750000,62.000000,90.800000,4263.800000,4.000000
max,594193.000000,72.000000,118.750000,8684.800000,6.000000


 # 1 - Prepare data for modeling 

In [ ]:
# In this cell, I split the data into training and test sets.
# I used stratify to keep the target distribution balanced in both sets.
# I assigned most of the data to training and kept 20% for testing.
# I created a summary table to compare the class counts in train and test.
# I displayed the number of No and Yes cases after the split.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=random_state,
    stratify=y,
)

split_summary = pd.DataFrame({
    "train_count": y_train.value_counts().sort_index(),
    "test_count": y_test.value_counts().sort_index(),
})
split_summary.index = ["No", "Yes"]
split_summary

,train_count,test_count
No,368301,92076
Yes,107054,26763


### data preprocessing pipeline


In [ ]:
# In this cell, I built the preprocessing pipeline for the dataset.
# I scaled the numerical features using StandardScaler.
# I encoded the categorical features using OneHotEncoder.
# I combined both transformations using ColumnTransformer.
# I prepared one preprocessing step that can be reused with all models.
numeric_pipeline = Pipeline([
    ("scaler", StandardScaler()),
])

categorical_pipeline = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_columns),
        ("cat", categorical_pipeline, categorical_columns),
    ],
    sparse_threshold=1.0,
)

roc_auc_scoring = "roc_auc"
validation_size = 0.2
tuning_cv = 3
kfold_splits = 5
search_jobs = 1
parallel_jobs = -1

def get_catboost_compute_params():
    try:
        from catboost.utils import get_gpu_device_count

        if get_gpu_device_count() > 0:
            return {"task_type": "GPU", "devices": "0"}
    except Exception:
        pass
    return {}

catboost_compute_params = get_catboost_compute_params()

def build_pipeline(model, sampler=None):
    steps = [("preprocessor", preprocessor)]
    if sampler is not None:
        steps.append(("sampler", sampler))
        return ImbPipeline(steps + [("model", model)])
    return Pipeline(steps + [("model", model)])
        


# compare imbalance handling options


In [ ]:
# In this cell, I compared different ways to handle the class imbalance problem.
# I took a smaller sample from the training data to make this comparison faster.
# I tested class_weight against SMOTEENN on the sample.
# I checked which method gives a better balance between performance and cost.
# I used this step to choose the imbalance strategy before running the full models.
imbalance_sample_size = min(len(X_train), 30_000)

if imbalance_sample_size < len(X_train):
    X_imbalance, _, y_imbalance, _ = train_test_split(
        X_train,
        y_train,
        train_size=imbalance_sample_size,
        random_state=random_state,
        stratify=y_train,
    )
else:
    X_imbalance = X_train.copy()
    y_imbalance = y_train.copy()

X_imbalance_train, X_imbalance_valid, y_imbalance_train, y_imbalance_valid = train_test_split(
    X_imbalance,
    y_imbalance,
    test_size=0.2,
    random_state=random_state,
    stratify=y_imbalance,
)

imbalance_experiments = {
    "class_weight=balanced": build_pipeline(
        LogisticRegression(
            max_iter=3000,
            solver="liblinear",
            class_weight="balanced",
            random_state=random_state,
        )
    ),
    "SMOTEENN": build_pipeline(
        LogisticRegression(
            max_iter=3000,
            solver="liblinear",
            random_state=random_state,
        ),
        sampler=SMOTEENN(random_state=random_state),
    ),
}

imbalance_rows = []
for strategy_name, pipeline in imbalance_experiments.items():
    start = time.perf_counter()
    pipeline.fit(X_imbalance_train, y_imbalance_train)
    fit_seconds = time.perf_counter() - start
    probabilities = pipeline.predict_proba(X_imbalance_valid)[:, 1]
    imbalance_rows.append(
        {
            "strategy": strategy_name,
            "roc_auc": roc_auc_score(y_imbalance_valid, probabilities),
            "fit_seconds": fit_seconds,
        }
    )

imbalance_strategy_results = (
    pd.DataFrame(imbalance_rows)
    .sort_values(["roc_auc", "fit_seconds"], ascending=[False, True])
    .reset_index(drop=True)
)
imbalance_strategy_results["roc_auc"] = imbalance_strategy_results["roc_auc"].round(4)
imbalance_strategy_results["fit_seconds"] = imbalance_strategy_results["fit_seconds"].round(2)

selected_imbalance_strategy = imbalance_strategy_results.loc[0, "strategy"]
imbalance_choice = "class_weight" if selected_imbalance_strategy == "class_weight=balanced" else "smoteenn"

imbalance_strategy_results
        


,strategy,roc_auc,fit_seconds
0,class_weight=balanced,0.9160,0.23
1,SMOTEENN,0.9143,86.96


In [ ]:
# In this cell, I confirmed the imbalance strategy for the full training run.
# I kept class_weight as the selected option because it is more efficient on the full dataset.
# I displayed a quick summary of the training data size and positive class rate.
# I used this output to show the final decision before starting the modeling step.
if imbalance_choice != "class_weight":
    print("SMOTEENN was competitive on the sample, but class_weight stays selected because it is much cheaper on 594k rows.")
    imbalance_choice = "class_weight"
else:
    print("Selected imbalance strategy for the full run: class_weight")

pd.DataFrame(
    {
        "row_count": [len(X_train)],
        "positive_rate": [round(y_train.mean(), 4)],
        "preprocessed_feature_count": [len(preprocessor.fit(X_train).get_feature_names_out())],
    }
)
        


Selected imbalance strategy for the full run: class_weight


,row_count,positive_rate,preprocessed_feature_count
0,475355,0.2252,49


# 2- modeling

### baseline models without cross-validation


In [ ]:
# In this cell, I split the training data again into baseline train and validation sets.
# I trained three different models: Logistic Regression, Random Forest, and CatBoost.
# I evaluated each model on the validation set using ROC AUC score.
# I measured the training time for each model as well.
# I sorted the results to see which model performed best.
X_baseline_train, X_baseline_valid, y_baseline_train, y_baseline_valid = train_test_split(
    X_train,
    y_train,
    test_size=validation_size,
    random_state=random_state,
    stratify=y_train,
)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        solver="liblinear",
        class_weight="balanced",
        random_state=random_state,
    ),
    "Random Forest": RandomForestClassifier(
        random_state=random_state,
        class_weight="balanced",
        n_estimators=150,
        n_jobs=parallel_jobs,
    ),
    "CatBoost": CatBoostClassifier(
        random_state=random_state,
        verbose=0,
        allow_writing_files=False,
        auto_class_weights="Balanced",
        iterations=150,
        thread_count=parallel_jobs,
        **catboost_compute_params,
    ),
}

baseline_rows = []

for model_name, model in models.items():
    start = time.perf_counter()
    candidate = clone(build_pipeline(model))
    candidate.fit(X_baseline_train, y_baseline_train)
    validation_probabilities = candidate.predict_proba(X_baseline_valid)[:, 1]
    baseline_rows.append(
        {
            "model": model_name,
            "validation_roc_auc": roc_auc_score(y_baseline_valid, validation_probabilities),
            "fit_seconds": time.perf_counter() - start,
        }
    )

baseline_score_lookup = {
    row["model"]: row["validation_roc_auc"]
    for row in baseline_rows
}

baseline_results = (
    pd.DataFrame(baseline_rows)
    .sort_values(["validation_roc_auc", "fit_seconds"], ascending=[False, True])
    .reset_index(drop=True)
)
baseline_results["validation_roc_auc"] = baseline_results["validation_roc_auc"].round(4)
baseline_results["fit_seconds"] = baseline_results["fit_seconds"].round(2)

baseline_results
        


,model,validation_roc_auc,fit_seconds
0,CatBoost,0.9147,15.91
1,Logistic Regression,0.9083,5.56
2,Random Forest,0.8918,1293.08


In [ ]:
# In this cell, I selected the best model from the baseline comparison.
# I took the validation ROC AUC score of the top model.
# I created a simple summary to show which model will be tuned next.
top_model_name = baseline_results.loc[0, "model"]
top_model_baseline_roc_auc = baseline_score_lookup[top_model_name]

pd.DataFrame(
    {
        "selected_for_tuning": [top_model_name],
        "baseline_validation_roc_auc": [round(top_model_baseline_roc_auc, 4)],
    }
)
        


,selected_for_tuning,baseline_validation_roc_auc
0,CatBoost,0.9147


### tune only the top baseline model


In [ ]:
# In this cell, I defined the hyperparameter grids for the candidate models.
# I ran GridSearchCV only on the top baseline model.
# I searched for the best parameter combination using cross-validation.
# I evaluated the tuned model on the validation set.
# I compared the tuned result with the baseline result.
# I kept the tuned configuration only if it performed better.
param_grids = {
    "Logistic Regression": {
        "model__C": [0.1, 1.0, 3.0, 10.0],
    },
    "Random Forest": {
        "model__n_estimators": [150, 250],
        "model__max_depth": [None, 12, 20],
        "model__min_samples_leaf": [1, 5, 20],
    },
    "CatBoost": {
        "model__iterations": [150, 250],
        "model__depth": [4, 6, 8],
        "model__learning_rate": [0.03, 0.07, 0.1],
    },
}

start = time.perf_counter()
top_model_search = GridSearchCV(
    estimator=build_pipeline(models[top_model_name]),
    param_grid=param_grids[top_model_name],
    scoring=roc_auc_scoring,
    cv=tuning_cv,
    n_jobs=search_jobs,
    refit=True,
    error_score="raise",
)
top_model_search.fit(X_baseline_train, y_baseline_train)

tuned_validation_probabilities = top_model_search.best_estimator_.predict_proba(X_baseline_valid)[:, 1]
tuned_validation_roc_auc = roc_auc_score(y_baseline_valid, tuned_validation_probabilities)

use_tuned_params = tuned_validation_roc_auc > top_model_baseline_roc_auc
selected_params = top_model_search.best_params_ if use_tuned_params else {}
selected_config = "tuned" if use_tuned_params else "baseline"
selected_validation_roc_auc = tuned_validation_roc_auc if use_tuned_params else top_model_baseline_roc_auc

tuning_results = pd.DataFrame(
    [
        {
            "model": top_model_name,
            "baseline_validation_roc_auc": round(top_model_baseline_roc_auc, 4),
            "best_gridsearch_cv_roc_auc": round(top_model_search.best_score_, 4),
            "tuned_validation_roc_auc": round(tuned_validation_roc_auc, 4),
            "selected_config": selected_config,
            "selected_validation_roc_auc": round(selected_validation_roc_auc, 4),
            "fit_seconds": round(time.perf_counter() - start, 2),
            "best_params": top_model_search.best_params_,
            "selected_params": selected_params,
        }
    ]
)

tuning_results
        


,model,baseline_validation_roc_auc,best_gridsearch_cv_roc_auc,tuned_validation_roc_auc,selected_config,selected_validation_roc_auc,fit_seconds,best_params,selected_params
0,CatBoost,0.9147,0.9147,0.9149,tuned,0.9149,189.81,"{'model__depth': 6, 'model__iterations': 250, ...","{'model__depth': 6, 'model__iterations': 250, ..."


In [ ]:
# In this cell, I prepared the final model using the selected configuration.
# I applied Stratified K-Fold cross-validation to check if the model is stable.
# I calculated ROC AUC, precision, recall, F1 score, and accuracy for each fold.
# I summarized the average cross-validation results.
# I trained the final model on the full training set.
# I tested the final model on the test set.
# I created the confusion matrix and displayed the final classification report.
best_model_name = top_model_name
best_params = selected_params

kfold = StratifiedKFold(
    n_splits=kfold_splits,
    shuffle=True,
    random_state=random_state,
)

kfold_rows = []

for fold_number, (train_index, valid_index) in enumerate(kfold.split(X_train, y_train), start=1):
    X_fold_train = X_train.iloc[train_index]
    X_fold_valid = X_train.iloc[valid_index]
    y_fold_train = y_train.iloc[train_index]
    y_fold_valid = y_train.iloc[valid_index]

    fold_model = clone(build_pipeline(models[best_model_name]))
    if best_params:
        fold_model.set_params(**best_params)
    fold_model.fit(X_fold_train, y_fold_train)

    fold_probabilities = fold_model.predict_proba(X_fold_valid)[:, 1]
    fold_predictions = fold_model.predict(X_fold_valid)

    kfold_rows.append(
        {
            "fold": fold_number,
            "roc_auc": roc_auc_score(y_fold_valid, fold_probabilities),
            "precision": precision_score(y_fold_valid, fold_predictions),
            "recall": recall_score(y_fold_valid, fold_predictions),
            "f1": f1_score(y_fold_valid, fold_predictions),
            "accuracy": accuracy_score(y_fold_valid, fold_predictions),
        }
    )

kfold_results = pd.DataFrame(kfold_rows).round(4)

kfold_summary = pd.DataFrame(
    [
        {
            "model": best_model_name,
            "selected_config": selected_config,
            "mean_cv_roc_auc": round(kfold_results["roc_auc"].mean(), 4),
            "std_cv_roc_auc": round(kfold_results["roc_auc"].std(ddof=0), 4),
            "mean_cv_precision": round(kfold_results["precision"].mean(), 4),
            "mean_cv_recall": round(kfold_results["recall"].mean(), 4),
            "mean_cv_f1": round(kfold_results["f1"].mean(), 4),
            "mean_cv_accuracy": round(kfold_results["accuracy"].mean(), 4),
        }
    ]
)

final_model = clone(build_pipeline(models[best_model_name]))
if best_params:
    final_model.set_params(**best_params)
final_model.fit(X_train, y_train)

test_probabilities = final_model.predict_proba(X_test)[:, 1]
test_predictions = final_model.predict(X_test)

final_test_results = pd.DataFrame(
    [
        {
            "model": best_model_name,
            "selected_config": selected_config,
            "selection_validation_roc_auc": round(selected_validation_roc_auc, 4),
            "test_roc_auc": round(roc_auc_score(y_test, test_probabilities), 4),
            "test_precision": round(precision_score(y_test, test_predictions), 4),
            "test_recall": round(recall_score(y_test, test_predictions), 4),
            "test_f1": round(f1_score(y_test, test_predictions), 4),
            "test_accuracy": round(accuracy_score(y_test, test_predictions), 4),
        }
    ]
)

confusion_df = pd.DataFrame(
    confusion_matrix(y_test, test_predictions),
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"],
)

print(classification_report(y_test, test_predictions, target_names=["No", "Yes"]))
display(kfold_results)
display(kfold_summary)
display(final_test_results)
confusion_df
        


              precision    recall  f1-score   support

          No       0.96      0.80      0.87     92076
         Yes       0.56      0.88      0.68     26763

    accuracy                           0.82    118839
   macro avg       0.76      0.84      0.78    118839
weighted avg       0.87      0.82      0.83    118839



,fold,roc_auc,precision,recall,f1,accuracy
0,1,0.9155,0.5609,0.8792,0.6849,0.8178
1,2,0.9157,0.5605,0.8779,0.6842,0.8175
2,3,0.9138,0.5570,0.8767,0.6812,0.8152
3,4,0.9135,0.5583,0.8745,0.6815,0.8159
4,5,0.9159,0.5590,0.8838,0.6849,0.8168


,model,selected_config,mean_cv_roc_auc,std_cv_roc_auc,mean_cv_precision,mean_cv_recall,mean_cv_f1,mean_cv_accuracy
0,CatBoost,tuned,0.9149,0.001,0.5591,0.8784,0.6833,0.8166


,model,selected_config,selection_validation_roc_auc,test_roc_auc,test_precision,test_recall,test_f1,test_accuracy
0,CatBoost,tuned,0.9149,0.9155,0.5584,0.882,0.6838,0.8163


,Predicted No,Predicted Yes
Actual No,73405,18671
Actual Yes,3158,23605


In [ ]:
# In this cell, I extracted the feature names from the preprocessing step.
# I calculated feature importance from the final trained model.
# I sorted the features to see which ones have the strongest effect.
# I created an artifacts folder to save the final outputs.
# I saved the feature importance table as a CSV file.
# I saved the final trained model as a joblib file.
# I displayed the top important features and the saved file paths.
preprocessed_feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
model_features_df = pd.DataFrame({"feature": preprocessed_feature_names})
model_step = final_model.named_steps["model"]

if hasattr(model_step, "coef_"):
    model_features_df["importance"] = np.ravel(model_step.coef_)
    model_features_df["abs_importance"] = model_features_df["importance"].abs()
    model_features_df = model_features_df.sort_values("abs_importance", ascending=False).reset_index(drop=True)
elif hasattr(model_step, "feature_importances_"):
    model_features_df["importance"] = model_step.feature_importances_
    model_features_df = model_features_df.sort_values("importance", ascending=False).reset_index(drop=True)

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

model_features_path = artifacts_dir / "model_features.csv"
model_features_df.to_csv(model_features_path, index=False)

model_path = artifacts_dir / "final_churn_model.joblib"
joblib.dump(
    {
        "model_name": best_model_name,
        "selected_config": selected_config,
        "best_params": best_params,
        "pipeline": final_model,
        "features": model_features_df["feature"].tolist(),
    },
    model_path,
)

saved_artifacts = pd.DataFrame(
    {
        "artifact": ["Model features", "Final model"],
        "path": [str(model_features_path), str(model_path)],
    }
)

display(model_features_df.head(20))
saved_artifacts
        


,feature,importance
0,num__tenure,20.528788
1,cat__internetservice_Fiber optic,15.596912
2,cat__contract_Month-to-month,13.788923
3,cat__contract_Two year,8.506974
4,num__totalcharges,7.558203
5,num__monthlycharges,6.687589
6,cat__paymentmethod_Electronic check,6.496365
7,cat__multiplelines_No,2.110904
8,cat__internetservice_No,1.460357
9,cat__streamingtv_No,1.399186


,artifact,path
0,Model features,artifacts/model_features.csv
1,Final model,artifacts/final_churn_model.joblib
